In [1]:
import time
import torch
from transformers import AutoModelForSeq2SeqLM, NllbTokenizer

# 1. Chargement du modèle et du tokenizer
MODEL_NAME = "bilalfaye/nllb-200-distilled-600M-wo-fr-en"

print("Chargement du modèle en cours...")
tokenizer = NllbTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Utiliser le GPU s'il est disponible pour accélérer les calculs
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Modèle chargé avec succès sur : {device.upper()}\n")

/opt/anaconda3/envs/tf_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chargement du modèle en cours...


Loading weights: 100%|██████████| 509/509 [00:00<00:00, 7623.53it/s]


Modèle chargé avec succès sur : CPU



In [ ]:

# 2. Fonction de traduction générique
def translate(text: str, src_lang: str, tgt_lang: str) -> str:
    """
    src_lang / tgt_lang codes pour NLLB:
    - Wolof: 'wol_Latn'
    - Français: 'fra_Latn'
    """
    t0 = time.time()

    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)

    # Génération du texte traduit
    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
        max_new_tokens=128
    )

    result = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
    elapsed = round(time.time() - t0, 2)

    return result



In [5]:

# 3. Tests de validation rapide

# A. Sens Wolof -> Français (Pour le Retrieval RAG)
phrases_wolof = [
    "dama beug wout kayitu juddu?",
    "Dama soxla doctoor ndax sama biir dafay metti lool.",
    "Jërejëf lool ci dimbal bi."
]
print(phrases_wolof)

['dama beug wout kayitu juddu?', 'Dama soxla doctoor ndax sama biir dafay metti lool.', 'Jërejëf lool ci dimbal bi.']


In [6]:

print("=== TEST : WOLOF -> FRANÇAIS ===")
for text in phrases_wolof:
    traduction, duree = translate(text, src_lang="wol_Latn", tgt_lang="fra_Latn")
    print(f"WO : {text}")
    print(f"FR : {traduction} ({duree}s)")
    print("-" * 40)


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== TEST : WOLOF -> FRANÇAIS ===


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


WO : dama beug wout kayitu juddu?
FR : Je veux un certificat de naissance? (0.96s)
----------------------------------------


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


WO : Dama soxla doctoor ndax sama biir dafay metti lool.
FR : J'ai besoin d'un médecin parce que mon ventre fait mal. (1.39s)
----------------------------------------
WO : Jërejëf lool ci dimbal bi.
FR : Merci beaucoup pour l'aide. (0.75s)
----------------------------------------


In [ ]:

# B. Sens Français -> Wolof (Pour la génération de réponse)
phrases_francais = [
    "Comment puis-je obtenir un extrait de naissance ?",
    "Merci beaucoup pour votre aide."
]

print("\n=== TEST : FRANÇAIS -> WOLOF ===")
for text in phrases_francais:
    traduction, duree = translate(text, src_lang="fra_Latn", tgt_lang="wol_Latn")
    print(f"FR : {text}")
    print(f"WO : {traduction} ({duree}s)")
    print("-" * 40)